# FPL Point Prediction Model — Training & Evaluation

Trains a LightGBM model (one per position) to predict `total_points` per player per gameweek.

**Pipeline:**
1. Build features from 9 seasons of data (57 features)
2. Run temporal cross-validation (expanding window, 6 folds)
3. Train final model on all data except holdout (2024-25)
4. Save model to disk for use in the RL environment

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore', category=UserWarning)

DATA_DIR = Path('../data')
MODEL_DIR = Path('../models/point_predictor')

SEASONS = [
    '2016-17', '2017-18', '2018-19', '2019-20', '2020-21',
    '2021-22', '2022-23', '2023-24', '2024-25',
]
HOLDOUT = '2024-25'

## 1. Build Features

In [2]:
from fpl_rl.prediction.id_resolver import IDResolver
from fpl_rl.prediction.feature_pipeline import FeaturePipeline

id_resolver = IDResolver(DATA_DIR)
pipeline = FeaturePipeline(DATA_DIR, id_resolver, SEASONS)

print('Building features across all seasons...')
df = pipeline.build()
print(f'\nResult: {df.shape[0]:,} rows x {df.shape[1]} columns')
print(f'Seasons: {sorted(df["season"].unique())}')
print(f'Positions: {dict(df["position"].value_counts())}')

Building features across all seasons...


Seasons:   0%|          | 0/9 [00:00<?, ?season/s]

Understat players:   0%|          | 0/683 [00:00<?, ?player/s]

Understat players:   0%|          | 0/647 [00:00<?, ?player/s]

Understat players:   0%|          | 0/624 [00:00<?, ?player/s]

...

Understat players:   0%|          | 0/737 [00:00<?, ?player/s]

Understat players:   0%|          | 0/778 [00:00<?, ?player/s]

Understat players:   0%|          | 0/865 [00:00<?, ?player/s]

Understat players:   0%|          | 0/784 [00:00<?, ?player/s]


Result: 215,087 rows x 63 columns
Seasons: ['2016-17', '2017-18', '2018-19', '2019-20', '2020-21', '2021-22', '2022-23', '2023-24', '2024-25']
Positions: {'MID': np.int64(74588), 'DEF': np.int64(61543), 'FWD': np.int64(23111), 'GK': np.int64(20082)}


In [3]:
# Quick data quality check
print('=== Data Quality ===')
print(f'Target (total_points) NaN: {df["target"].isna().sum()}')
print(f'Position NaN: {df["position"].isna().sum()}')
print(f'\nTarget distribution:')
print(df['target'].describe().round(2))

# Feature NaN rates
feature_cols = [c for c in df.columns if c not in {'code','element','season','GW','position','target','total_points'}]
nan_pct = (df[feature_cols].isna().mean() * 100).sort_values(ascending=False)
print(f'\nFeature NaN rates (top 15):')
print(nan_pct.head(15).round(1))

=== Data Quality ===
Target (total_points) NaN: 0
Position NaN: 35763

Target distribution:
count    215087.00
mean          1.31
std           2.53
min          -7.00
25%           0.00
50%           0.00
75%           2.00
max          31.00
Name: target, dtype: float64

Feature NaN rates (top 15):
prev_prog_dist_per90     100.0
prev_pass_cmp_pct        100.0
prev_blocks_per90        100.0
prev_tkl_int_per90       100.0
prev_gls_per90            52.4
prev_sot_per90            52.4
prev_xg_per90             49.5
prev_xa_per90             49.5
prev_npxg_per90           49.5
prev_key_passes_per90     49.5
prev_shots_per90          49.5
prev_minutes              45.8
xg_rolling_5              34.0
xg_rolling_10             34.0
xa_rolling_3              34.0
dtype: float64


In [4]:
# Drop rows without position or target
df = df.dropna(subset=['position', 'target'])
print(f'After dropping NaN target/position: {len(df):,} rows')

After dropping NaN target/position: 179,324 rows


## 2. Temporal Cross-Validation

In [5]:
from fpl_rl.prediction.evaluation import TemporalCV

cv = TemporalCV(min_train_seasons=2, holdout=HOLDOUT)
folds = cv.generate_folds(df)
print(f'{len(folds)} folds generated\n')

for i, (train, val, test) in enumerate(folds):
    train_seasons = sorted(train['season'].unique())
    test_season = test['season'].iloc[0]
    print(f'Fold {i+1}: train={train_seasons} ({len(train):,}), '
          f'val={len(val):,}, test={test_season} ({len(test):,})')

6 folds generated

Fold 1: train=['2016-17', '2017-18'] (17,889), val=2,292, test=2018-19 (14,098)
Fold 2: train=['2016-17', '2017-18', '2018-19'] (31,425), val=2,854, test=2019-20 (18,308)
Fold 3: train=['2016-17', '2017-18', '2018-19', '2019-20'] (48,356), val=4,231, test=2020-21 (22,889)
Fold 4: train=['2016-17', '2017-18', '2018-19', '2019-20', '2020-21'] (70,261), val=5,215, test=2021-22 (23,230)
Fold 5: train=['2016-17', '2017-18', '2018-19', '2019-20', '2020-21', '2021-22'] (93,020), val=5,686, test=2022-23 (24,957)
Fold 6: train=['2016-17', '2017-18', '2018-19', '2019-20', '2020-21', '2021-22', '2022-23'] (117,689), val=5,974, test=2023-24 (28,742)


In [6]:
print('Running temporal CV (this may take a minute)...\n')
results = cv.evaluate(df, params={'n_estimators': 500, 'verbose': -1})

print(f'Overall MAE:  {results["mae"]:.3f}')
print(f'Overall RMSE: {results["rmse"]:.3f}')
print(f'\nPer-position MAE:')
for pos, mae in sorted(results['per_position_mae'].items()):
    print(f'  {pos}: {mae:.3f}')

print(f'\nPer-fold results:')
for fold in results['fold_results']:
    print(f'  Fold {fold["fold"]} ({fold["test_season"]}): '
          f'MAE={fold["mae"]:.3f}, RMSE={fold["rmse"]:.3f}, n={fold["n_test"]:,}')

Running temporal CV (this may take a minute)...

Overall MAE:  1.241
Overall RMSE: 2.186

Per-position MAE:
  DEF: 1.349
  FWD: 1.428
  GK: 0.894
  MID: 1.188

Per-fold results:
  Fold 1 (2018-19): MAE=1.619, RMSE=2.495, n=14,098
  Fold 2 (2019-20): MAE=1.463, RMSE=2.330, n=18,308
  Fold 3 (2020-21): MAE=1.261, RMSE=2.229, n=22,889
  Fold 4 (2021-22): MAE=1.255, RMSE=2.230, n=23,230
  Fold 5 (2022-23): MAE=1.114, RMSE=2.052, n=24,957
  Fold 6 (2023-24): MAE=0.997, RMSE=1.958, n=28,742


## 3. Train Final Model

Train on all seasons except the holdout (2024-25), using the last 8 GWs of 2023-24 as validation for early stopping.

In [7]:
from fpl_rl.prediction.model import PointPredictor

# Split train / val
train_seasons = [s for s in SEASONS if s != HOLDOUT]
train_full = df[df['season'].isin(train_seasons)]

last_season = train_seasons[-1]
last_data = train_full[train_full['season'] == last_season]
val_cutoff = last_data['GW'].max() - 8

val_mask = (train_full['season'] == last_season) & (train_full['GW'] > val_cutoff)
val_df = train_full[val_mask]
train_df = train_full[~val_mask]

print(f'Training: {len(train_df):,} rows ({sorted(train_df["season"].unique())})')
print(f'Validation: {len(val_df):,} rows (last 8 GWs of {last_season})')

predictor = PointPredictor(
    params={'n_estimators': 500, 'verbose': -1},
    early_stopping_rounds=50,
)
train_results = predictor.train(train_df, val_df)

print(f'\nTraining MAE per position:')
for pos, mae in sorted(train_results.items()):
    print(f'  {pos}: {mae:.3f}')

Training: 145,605 rows (['2016-17', '2017-18', '2018-19', '2019-20', '2020-21', '2021-22', '2022-23', '2023-24'])
Validation: 6,800 rows (last 8 GWs of 2023-24)

Training MAE per position:
  DEF: 1.090
  FWD: 1.040
  GK: 0.593
  MID: 1.019


In [8]:
# Feature importance (top 20)
fi = predictor.feature_importance()
print('Top 20 features by importance (gain):\n')
print(fi.head(20).to_string(index=False))

Top 20 features by importance (gain):

             feature    importance
      mins_rolling_3 293328.433462
       ict_rolling_3 153833.916017
       pts_rolling_3  60784.260128
        xg_rolling_3  52820.659708
       selected_norm  45455.882506
               value  39055.318385
      ict_rolling_10  36865.459237
              is_dgw  34637.495011
 opp_pts_conceded_r5  30609.282893
        xa_rolling_3  27112.577118
      season_avg_pts  26088.161532
   season_total_mins  23683.938828
opp_defence_strength  22757.453249
    threat_rolling_5  21346.383266
        xa_rolling_5  20870.526082
 opp_attack_strength  20382.760299
       ict_rolling_5  20359.157513
                 fdr  19628.176637
          mins_std_5  19250.077614
        xg_rolling_5  17863.196578


## 4. Evaluate on Holdout (2024-25)

In [9]:
holdout_df = df[df['season'] == HOLDOUT]

if holdout_df.empty:
    print(f'No holdout data for {HOLDOUT}')
else:
    preds = predictor.predict(holdout_df)
    actuals = holdout_df['target'].values
    errors = np.abs(preds - actuals)

    print(f'Holdout ({HOLDOUT}): {len(holdout_df):,} rows')
    print(f'MAE:  {errors.mean():.3f}')
    print(f'RMSE: {np.sqrt(np.mean((preds - actuals)**2)):.3f}')

    print(f'\nPer-position holdout MAE:')
    for pos in ['GK', 'DEF', 'MID', 'FWD']:
        mask = holdout_df['position'].values == pos
        if mask.any():
            print(f'  {pos}: {errors[mask].mean():.3f} (n={mask.sum():,})')

Holdout (2024-25): 26,919 rows
MAE:  1.005
RMSE: 1.923

Per-position holdout MAE:
  GK: 0.764 (n=2,827)
  DEF: 1.003 (n=9,024)
  MID: 1.016 (n=12,084)
  FWD: 1.189 (n=2,984)


## 5. Save Model

In [10]:
predictor.save(MODEL_DIR)
print(f'Model saved to {MODEL_DIR.resolve()}')
print(f'Files: {[f.name for f in MODEL_DIR.iterdir()]}')

# Verify load roundtrip
loaded = PointPredictor.load(MODEL_DIR)
test_preds = loaded.predict(holdout_df.head(10))
print(f'\nLoad roundtrip OK — sample predictions: {test_preds.round(2)}')

Model saved to C:\Users\alexa\OneDrive\Documents\FPL-RL\models\point_predictor
Files: ['DEF.lgb', 'feature_names.json', 'FWD.lgb', 'GK.lgb', 'metadata.json', 'MID.lgb']

Load roundtrip OK — sample predictions: [0.66 0.12 0.09 0.05 0.05 0.1  0.1  0.05 0.04 0.05]


## 6. Quick Sanity Checks

Spot-check predictions for well-known players.

In [11]:
if not holdout_df.empty:
    holdout_with_preds = holdout_df.copy()
    holdout_with_preds['predicted'] = preds
    holdout_with_preds['error'] = errors

    # Per-GW average prediction vs actual
    gw_summary = holdout_with_preds.groupby('GW').agg(
        avg_actual=('target', 'mean'),
        avg_predicted=('predicted', 'mean'),
        mae=('error', 'mean'),
    ).round(3)
    print('Per-GW summary (holdout):')
    print(gw_summary.to_string())

    # Top predicted players in a sample GW
    sample_gw = holdout_with_preds['GW'].median()
    gw_data = holdout_with_preds[holdout_with_preds['GW'] == sample_gw]
    top = gw_data.nlargest(10, 'predicted')[['code', 'position', 'predicted', 'target']]
    top['name'] = top['code'].apply(id_resolver.player_name)
    print(f'\nTop 10 predicted in GW {int(sample_gw)}:')
    print(top[['name', 'position', 'predicted', 'target']].to_string(index=False))

Per-GW summary (holdout):
    avg_actual  avg_predicted    mae
GW                                  
1        1.317          1.385  1.236
2        1.367          1.337  1.006
3        1.140          1.304  0.880
4        1.235          1.287  1.055
5        1.212          1.273  1.035
6        1.142          1.258  0.995
7        1.311          1.251  1.056
8        1.165          1.223  1.006
9        1.151          1.212  0.999
10       1.150          1.229  0.946
11       1.201          1.235  1.006
12       1.228          1.228  1.130
13       1.200          1.201  1.048
14       1.248          1.186  1.127
15       1.127          1.185  1.000
16       1.168          1.219  1.001
17       1.312          1.174  1.167
18       1.235          1.210  1.011
19       1.176          1.197  1.028
20       1.080          1.209  0.948
21       1.122          1.180  0.931
22       1.167          1.161  1.087
23       1.102          1.186  0.978
24       1.256          1.215  1.085
25       1.2

---

## Usage in the RL Environment

```python
from fpl_rl.env.fpl_env import FPLEnv

env = FPLEnv(
    season='2024-25',
    predictor_model_dir='models/point_predictor',
)
# features[18] in observations now contains predicted points / 15.0
```